In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tok = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cuda"
)

print("model loaded")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

model loaded


In [2]:
def prompt_of_len(n):
    text = ("I am learning AI data center operations, GPU inference, and model serving. " * (n + 1))
    ids = tok(text, add_special_tokens=False)["input_ids"][:n]
    return tok.decode(ids)

In [3]:
import time

results = {}
for context in [128, 512, 2048]:
    prompt = prompt_of_len(context)
    t0 = time.time()
    out = model.generate(**tok(prompt, return_tensors="pt"), max_new_tokens=32)
    results[context] = time.time() - t0
    print(context, results[context])

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:2636: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


128 4.572084903717041
512 2.112415075302124
2048 2.436420202255249


In [4]:
results = {}
for context in [2048, 512, 128]:
    prompt = prompt_of_len(context)
    t0 = time.time()
    out = model.generate(**tok(prompt, return_tensors="pt"), max_new_tokens=32)
    results[context] = time.time() - t0
    print(context, results[context])

2048 4.450009346008301
512 3.1344754695892334
128 1.7687952518463135


In [5]:
# throwaway warm-up, discarded, absorbs the one-time cost before real timing
_ = model.generate(**tok(prompt_of_len(64), return_tensors="pt"), max_new_tokens=8)

results = {}
for context in [128, 512, 2048]:
    prompt = prompt_of_len(context)
    t0 = time.time()
    out = model.generate(**tok(prompt, return_tensors="pt"), max_new_tokens=32)
    results[context] = time.time() - t0
    print(context, results[context])

128 1.68766450881958
512 1.7154850959777832
2048 2.31858229637146


In [6]:
assert results[128] < results[512] < results[2048], (
    f"expected latency to climb with context length, got {results}"
)
print("GREEN CHECK: PASS")

GREEN CHECK: PASS


In [7]:
print("W3D5 Bug Lab - Warm-up Fix Results")
print(f"128 tokens:  {results[128]:.3f} s")
print(f"512 tokens:  {results[512]:.3f} s")
print(f"2048 tokens: {results[2048]:.3f} s")

assert results[128] < results[512] < results[2048], (
    f"expected latency to climb with context length, got {results}"
)

print("GREEN CHECK: PASS")

W3D5 Bug Lab - Warm-up Fix Results
128 tokens:  1.688 s
512 tokens:  1.715 s
2048 tokens: 2.319 s
GREEN CHECK: PASS
